In [1]:
# %% CELL — ANALYSIS of the returned expert CSVs  (+ link with the rule-based ADCD scores)
# Metrics in three groups (as in Table tab:expert-metrics):
#   pathology findings | missing-tooth statements | global (both pooled) + clinically relevant reports
import glob
import numpy as np, pandas as pd
from scipy.stats import wilcoxon, spearmanr
from pathlib import Path

OUTPUT_DIR = r"Final Evaluations"
rd = lambda p: pd.read_csv(p, sep=None, engine="python", encoding="utf-8-sig")   # , or ; separated

KEY = rd(f"{OUTPUT_DIR}/expert_key_DO_NOT_UPLOAD.csv")               # case/report -> mode, image
if "rater3" not in set(KEY.rater):                                   
    KEY = pd.concat([KEY, KEY[KEY.rater == "rater2"].assign(
        rater="rater3", case=lambda d: d.case.str.replace("rater2_", "rater3_", regex=False))])
EV  = pd.concat([rd(f) for f in sorted(glob.glob(f"{OUTPUT_DIR}/evaluation_rater*.csv"))], ignore_index=True)
EV  = EV.merge(KEY, on=["rater", "case", "report"])
print(f"{EV.rater.nunique()} rater(s) | {EV.image.nunique()} images | incomplete reports: {(EV.complete == 0).sum()}")
EV  = EV[EV.complete == 1].copy()
# design: rater3 = one expert who rated all 100 images; rater2 = the second assessment
#         (two experts, 50 images each, merged into one file) -> two ratings per image
EXPERT = {"rater3": "expert_1", "rater2": "expert_2"}
EV["expert"] = EV.rater.map(EXPERT)
assert EV.expert.notna().all(), f"unknown rater(s): {set(EV.rater) - set(EXPERT)}"
print("ratings per image:", EV.groupby("image").expert.nunique().value_counts().to_dict())

CNT   = ["TP", "LOC", "FP", "FN", "mTP", "mFP", "mFN"]
ORDER = ["mode_a", "mode_c", "mode_d"]
LABEL = {"mode_a": "Mode A (baseline)", "mode_c": "Mode C", "mode_d": "Mode D (framework)"}
RELEVANT = ["as_is", "minor"]                                        # highly relevant + minor corrections

GROUPS = {                                                           # group -> metric names (table order)
    "Pathology findings": ["Precision", "Recall", "Detection recall", "Hallucination rate ↓",
                           "Localization error rate ↓"],
    "Missing-tooth statements": ["MT precision", "MT recall", "MT hallucination rate ↓"],
    "Global": ["Global precision", "Global recall", "Global hallucination rate ↓",
               "Clinically relevant reports (%)"],
}

def metrics(c):
    """c: summed counts -> metrics (micro average when c is a sum over reports)."""
    d = lambda a, b: a / b if b else np.nan
    rep = c.TP + c.LOC + c.FP                       # reported pathologies
    S   = rep + c.mTP + c.mFP                       # all statements in the reports
    G   = c.TP + c.LOC + c.FN + c.mTP + c.mFN       # all visible pathologies + absent teeth
    return pd.Series({
        "Precision":                   d(c.TP, rep),
        "Recall":                      d(c.TP, c.TP + c.LOC + c.FN),
        "Detection recall":            d(c.TP + c.LOC, c.TP + c.LOC + c.FN),
        "Hallucination rate ↓":        d(c.FP, rep),
        "Localization error rate ↓":   d(c.LOC, c.TP + c.LOC),
        "MT precision":                d(c.mTP, c.mTP + c.mFP),
        "MT recall":                   d(c.mTP, c.mTP + c.mFN),
        "MT hallucination rate ↓":     d(c.mFP, c.mTP + c.mFP),
        "Global precision":            d(c.TP + c.mTP, S),
        "Global recall":               d(c.TP + c.mTP, G),
        "Global hallucination rate ↓": d(c.FP + c.mFP, S),
    })

# 1) micro metrics per mode (all reports of all experts pooled) + per expert
tab = EV.groupby("mode")[CNT].sum().apply(metrics, axis=1).T[ORDER] * 100
tab.loc["Clinically relevant reports (%)"] = (EV.acceptability.isin(RELEVANT).groupby(EV["mode"]).mean() * 100)[ORDER].values
per_expert = EV.groupby(["expert", "mode"])[CNT].sum().apply(metrics, axis=1) * 100
per_expert["Clinically relevant reports (%)"] = EV.acceptability.isin(RELEVANT).groupby([EV.expert, EV["mode"]]).mean() * 100

# 2) paired Wilcoxon tests per image (counts averaged over the experts of the image, metric per image)
img_m = EV.groupby(["image", "mode"])[CNT].mean().apply(metrics, axis=1) * 100
img_m["Clinically relevant reports (%)"] = EV.acceptability.isin(RELEVANT).groupby([EV.image, EV["mode"]]).mean() * 100
def p_value(metric, a, b):
    w = img_m[metric].unstack("mode")[[a, b]].dropna()
    if len(w) < 6 or (w[a] == w[b]).all():
        return np.nan
    return wilcoxon(w[a], w[b]).pvalue
PAIRS = [("mode_a", "mode_d"), ("mode_c", "mode_d"), ("mode_a", "mode_c")]
for a, b in PAIRS:
    tab[f"p {a[-1].upper()} vs {b[-1].upper()}"] = [p_value(m, a, b) for m in tab.index]

pd.set_option("display.width", 220)
print("\n=== Expert evaluation (micro average, %) ===")
for g, rows in GROUPS.items():
    print(f"\n-- {g} --")
    print(tab.loc[rows].rename(columns=LABEL).round(3).to_string())
print("\n=== Per expert (%) ===")
print(per_expert.round(1).to_string())

# 3) inter-rater agreement on the images rated by both experts
#    ICC(1,1): the second rating comes from two different experts -> one-way random-effects model
def icc11(x):                                   # x: n_items x k ratings
    x = np.asarray(x, float); n, k = x.shape
    msb = k * ((x.mean(1) - x.mean()) ** 2).sum() / (n - 1)
    msw = ((x - x.mean(1, keepdims=True)) ** 2).sum() / (n * (k - 1))
    return (msb - msw) / (msb + (k - 1) * msw)
if EV.expert.nunique() == 2:
    wide = EV.pivot_table(index=["image", "mode"], columns="expert", values=CNT).dropna()
    print(f"\n=== Inter-rater agreement ({len(wide)} reports rated by both experts) ===")
    for c in CNT:
        w = wide[c]
        if w.values.std() > 0:
            print(f"  {c:<4} ICC(1,1) = {icc11(w.values):.3f}")
    from sklearn.metrics import cohen_kappa_score
    lv = {"reject": 0, "major": 1, "minor": 2, "as_is": 3}
    ak = EV.pivot_table(index=["image", "mode"], columns="expert", values="acceptability", aggfunc="first").dropna()
    print(f"  Clinical relevance, quadratically weighted kappa = "
          f"{cohen_kappa_score(ak.expert_1.map(lv), ak.expert_2.map(lv), weights='quadratic'):.3f}  (n = {len(ak)})")

# 4) agreement with the rule-based ADCD scores (step 13, rule_based_counts.csv)
rb_path = f"{OUTPUT_DIR}/rule_based_counts.csv"
if Path(rb_path).exists():
    RB = rd(rb_path)
    RB["rb_prec"] = 100 * RB.TP / (RB.TP + RB.FP).replace(0, np.nan)
    j = img_m.reset_index().merge(RB[["image", "mode", "rb_prec"]], on=["image", "mode"]).dropna(
        subset=["Precision", "rb_prec"])
    if len(j) > 5:
        r, p = spearmanr(j["Precision"], j["rb_prec"])
        print(f"\n=== Expert precision vs rule-based ADCD precision: Spearman r = {r:.2f} (p = {p:.3g}, n = {len(j)})")

# 5) save: CSV + LaTeX rows grouped like the definitions table
tab.to_csv(f"{OUTPUT_DIR}/expert_metrics_table.csv")
NAME = {"MT precision": "Precision", "MT recall": "Recall", "MT hallucination rate ↓": "Hallucination rate",
        "Global precision": "Precision", "Global recall": "Recall", "Global hallucination rate ↓": "Hallucination rate",
        "Hallucination rate ↓": "Hallucination rate", "Localization error rate ↓": "Localization error rate"}
LOWER = {m for m in tab.index if "↓" in m}
def fmt_p(p):
    return "--" if pd.isna(p) else ("$<$0.001" if p < 0.001 else f"{p:.3f}")
lines = []
for g, rows in GROUPS.items():
    lines.append(rf"\multicolumn{{7}}{{l}}{{\textit{{{g}}}}} \\")
    for m in rows:
        v = tab.loc[m, ORDER]
        best = v.idxmin() if m in LOWER else v.idxmax()
        cells = [(rf"\textbf{{{v[k]:.1f}}}" if k == best else f"{v[k]:.1f}") for k in ORDER]
        name = NAME.get(m, m.replace(" (%)", "")) + (r" $\downarrow$" if m in LOWER else "")
        ps = [fmt_p(tab.loc[m, f"p {a[-1].upper()} vs {b[-1].upper()}"]) for a, b in PAIRS]
        lines.append(f"{name} & " + " & ".join(cells) + " & " + " & ".join(ps) + r" \\")
    lines.append(r"\midrule")

2 rater(s) | 100 images | incomplete reports: 0
ratings per image: {2: 100}

=== Expert evaluation (micro average, %) ===

-- Pathology findings --
mode                       Mode A (baseline)  Mode C  Mode D (framework)  p A vs D  p C vs D  p A vs C
Precision                             46.296  76.038              74.421     0.000     0.390     0.000
Recall                                 1.400  50.031              49.140     0.000     0.538     0.000
Detection recall                       1.400  56.611              54.939     0.000     0.088     0.000
Hallucination rate ↓                  53.704  13.962              16.795     0.000     0.101     0.000
Localization error rate ↓              0.000  11.623              10.557     0.001     0.183     0.001

-- Missing-tooth statements --
mode                     Mode A (baseline)  Mode C  Mode D (framework)  p A vs D  p C vs D  p A vs C
MT precision                           0.0  86.667              86.196       NaN     0.443       NaN
